# Profile Culture-Organism Grain

## Purpose

This notebook validates the source-level culture-organism fact and profiles
the organism vocabulary before terminology standardization.

The upstream culture table contains one row per culture order. A positive
culture can contain one or more organism labels, so
`fct_culture_organism` uses a more detailed grain:

> one source culture-organism combination

The source does not provide a verified isolate identifier. Repeated
antibiotic records are therefore flagged for investigation rather than
being interpreted automatically as separate isolates.

## Models evaluated

- `workspace.micro_dev.fct_culture`
- `workspace.micro_dev.fct_culture_organism`

## Validation questions

1. Is the culture-organism key unique?
2. How many culture orders contain at least one organism?
3. How often do culture-organism groups contain repeated antibiotic records?
4. Which source organism labels are most common?
5. How large is the complete source organism vocabulary?

In [1]:
%sql
select
    count(*) as culture_organism_rows,
    count(distinct culture_organism_key) as distinct_keys,
    count(distinct culture_key) as cultures_with_organisms,
    sum(
        case
            when has_repeated_antibiotic_records then 1
            else 0
        end
    ) as groups_with_repeated_antibiotic_records
from workspace.micro_dev.fct_culture_organism;

,culture_organism_rows,distinct_keys,cultures_with_organisms,groups_with_repeated_antibiotic_records
0,133246,133246,118767,4546


## Culture-organism grain validation

Expected results for ARMD release `2025-04-11`:

| Measure | Expected |
|---|---:|
| Culture-organism rows | 133,246 |
| Distinct culture-organism keys | 133,246 |
| Cultures with organisms | 118,767 |
| Groups with repeated antibiotic records | 4,546 |

Equal row and key counts confirm the intended fact-table grain.

The repeated-antibiotic flag is a data-quality signal. It indicates that
the same source culture-organism contains more result rows than distinct
antibiotic labels; it does not, by itself, prove that the records belong
to separate isolates.

In [2]:
%sql
select
    organism,
    count(*) as culture_count
from workspace.micro_dev.fct_culture_organism
group by organism
order by culture_count desc
limit 50;

,organism,culture_count
0,ESCHERICHIA COLI,61348
1,KLEBSIELLA PNEUMONIAE,10369
2,ENTEROCOCCUS SPECIES,9462
3,STAPHYLOCOCCUS AUREUS,8415
4,PSEUDOMONAS AERUGINOSA,5745
5,PROTEUS MIRABILIS,4750
6,MUCOID PSEUDOMONAS AERUGINOSA,3324
7,STREPTOCOCCUS AGALACTIAE (GROUP B),2743
8,COAG NEGATIVE STAPHYLOCOCCUS,2230
9,KLEBSIELLA OXYTOCA,1671


## Most common source organism labels

This frequency table uses the organism names exactly as supplied by ARMD.

It is intended as a source-profile view, not as a standardized taxonomy.
High-frequency labels help prioritize which organisms should receive
reviewed mappings first, while uncommon labels remain visible for later
review.

In [3]:
%sql
select
    count(distinct organism) as distinct_source_organism_names
from workspace.micro_dev.fct_culture_organism;

,distinct_source_organism_names
0,314


## Source vocabulary size

The current release contains **314 distinct source organism labels**.

This is substantially larger than the initial reporting set because the
source includes species, complexes, groups, historical names, phenotype
qualifiers, morphotypes, and broad descriptive categories.

In [4]:
%sql
select
    organism,
    count(*) as culture_count
from workspace.micro_dev.fct_culture_organism
group by organism
order by organism;

,organism,culture_count
0,ABIOTROPHIA SPECIES (OR GRANULICATELLA SPECIES),6
1,ACHROMOBACTER SPECIES,31
2,ACHROMOBACTER XYLOSOXIDANS,489
3,ACID FAST BACILLI,2
4,ACID FAST BACILLI DETECTED IN LIQUID MEDIUM,3
5,ACINETOBACTER BAUMANNII,48
6,ACINETOBACTER BAUMANNII COMPLEX,96
7,ACINETOBACTER HAEMOLYTICUS,2
8,ACINETOBACTER LWOFFI,42
9,ACINETOBACTER SPECIES,152


## Complete vocabulary for terminology review

The alphabetic listing is the audit source for future mapping decisions.

The notebook display may show only the first portion of a large result,
but the SQL query returns the full set of source labels. Any mapping seed
should be created from a reviewed export of this vocabulary rather than
from ad hoc string replacement.

## Conclusion

`fct_culture_organism` has the expected unique grain and preserves the
source terminology needed for downstream reconciliation.

The profile also demonstrates why organism standardization must be
explicit and conservative: formatting variants can be normalized, but
species, complexes, groups, phenotype labels, and morphotypes should not
be collapsed without a documented analytical rule.